# Fake Product Review Detection
## Part 3 — Model Saving, Advanced Evaluation & Cross-Validation

This notebook covers everything needed to move from a trained model to a deployable artifact:
1. Retrain all 6 pipelines from the preprocessed CSV
2. Evaluate with Accuracy, Precision, Recall, and F1-Score
3. K-Fold Cross-Validation to confirm results aren't a lucky split
4. Save the best model as a `.pkl` file using `joblib`
5. Verify the saved model loads and predicts correctly

## 1. Imports & Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import string, warnings, os
import joblib
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression

warnings.filterwarnings('ignore')

# Directory for saved models
os.makedirs('models', exist_ok=True)
print('All libraries imported. models/ directory ready.')

## 2. Load Preprocessed Dataset

In [ ]:
df = pd.read_csv('Preprocessed_Fake_Reviews_Detection_Dataset.csv')
df.dropna(inplace=True)
print('Shape:', df.shape)
print('Label distribution:')
print(df['label'].value_counts())

## 3. Shared Preprocessing Function

In [ ]:
def text_process(review):
    """Strip residual punctuation and tokenise (text already lemmatised in Part 1)."""
    nopunc = [char for char in str(review) if char not in string.punctuation]
    return ''.join(nopunc).split()

## 4. Train / Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df['text_'], df['label'],
    test_size=0.35, random_state=42, stratify=df['label']
)
print('Train:', len(X_train), '| Test:', len(X_test))
print('Train label counts:')
print(pd.Series(y_train).value_counts())

## 5. Train All 6 Models + Collect Full Metrics

In [ ]:
def build_pipeline(classifier):
    return Pipeline([
        ('bow',   CountVectorizer(analyzer=text_process)),
        ('tfidf', TfidfTransformer()),
        ('clf',   classifier)
    ])

MODELS = {
    'Logistic Regression':     LogisticRegression(max_iter=1000, random_state=42),
    'Support Vector Machine':  SVC(random_state=42),
    'Random Forest':           RandomForestClassifier(n_estimators=100, random_state=42),
    'Decision Tree':           DecisionTreeClassifier(random_state=42),
    'Naive Bayes':             MultinomialNB(),
    'KNN (k=5)':              KNeighborsClassifier(n_neighbors=5),
}

results = []
trained_pipelines = {}

for name, clf in MODELS.items():
    pipe = build_pipeline(clf)
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)
    trained_pipelines[name] = pipe

    results.append({
        'Model':     name,
        'Accuracy':  round(accuracy_score(y_test, preds) * 100, 2),
        'Precision': round(precision_score(y_test, preds, pos_label='CG') * 100, 2),
        'Recall':    round(recall_score(y_test, preds, pos_label='CG') * 100, 2),
        'F1-Score':  round(f1_score(y_test, preds, pos_label='CG') * 100, 2),
    })
    print(f'✓ {name}')

results_df = pd.DataFrame(results).sort_values('F1-Score', ascending=False).reset_index(drop=True)
print('\nMetrics Table (sorted by F1-Score):')
results_df

## 6. Visual Metric Comparison

> **Why F1 matters more than Accuracy:** if 60% of reviews are genuine (OR), a model that always predicts OR scores 60% accuracy while detecting 0 fake reviews. F1-Score balances precision and recall and is the honest measure for imbalanced detection tasks.

In [ ]:
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
axes = axes.flatten()
colors = ['#2196F3', '#4CAF50', '#FF9800', '#E91E63']

for ax, metric, color in zip(axes, metrics, colors):
    sorted_df = results_df.sort_values(metric, ascending=True)
    bars = ax.barh(sorted_df['Model'], sorted_df[metric], color=color, alpha=0.85)
    ax.set_xlabel(f'{metric} (%)')
    ax.set_title(f'{metric} Comparison', fontweight='bold')
    ax.set_xlim(40, 100)
    for bar, val in zip(bars, sorted_df[metric]):
        ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
                f'{val}%', va='center', fontsize=9)

plt.suptitle('Fake Review Detection — All Model Metrics', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('models/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Confusion Matrices for All Models

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for ax, (name, pipe) in zip(axes, trained_pipelines.items()):
    preds = pipe.predict(X_test)
    cm = confusion_matrix(y_test, preds, labels=['CG', 'OR'])
    disp = ConfusionMatrixDisplay(cm, display_labels=['CG (Fake)', 'OR (Genuine)'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(name, fontweight='bold')

plt.suptitle('Confusion Matrices', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 8. K-Fold Cross-Validation

A single `train_test_split` can give lucky or unlucky results depending on which rows land in which split. **Stratified 5-Fold CV** trains and evaluates on 5 different splits and reports the mean ± standard deviation — this proves your scores are stable, not a fluke.

In [ ]:
print('Running 5-Fold Stratified Cross-Validation...')
print('(This may take a few minutes for SVM and Random Forest)\n')

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = []

for name, clf in MODELS.items():
    pipe = build_pipeline(clf)
    scores = cross_val_score(pipe, df['text_'], df['label'],
                             cv=cv, scoring='f1_weighted', n_jobs=-1)
    cv_results.append({
        'Model': name,
        'CV F1 Mean (%)': round(scores.mean() * 100, 2),
        'CV F1 Std (±%)': round(scores.std() * 100, 2),
    })
    print(f'{name:30s}  F1: {scores.mean()*100:.2f}% ± {scores.std()*100:.2f}%')

cv_df = pd.DataFrame(cv_results).sort_values('CV F1 Mean (%)', ascending=False)
cv_df

## 9. Save All Trained Pipelines

### Why `joblib` and not `pickle`?
`joblib` is faster and more memory-efficient for NumPy arrays and sparse matrices (which is exactly what BoW/TF-IDF produces). It's the standard recommended by scikit-learn.

### Why save the full Pipeline?
The Pipeline bundles the `CountVectorizer` + `TfidfTransformer` + classifier together. When you load it later, calling `.predict(["your text here"])` automatically runs the full transformation chain — you don't need to manage the vectorizers separately.

In [ ]:
# Save every trained pipeline
for name, pipe in trained_pipelines.items():
    filename = name.lower().replace(' ', '_').replace('(', '').replace(')', '').replace('=', '') + '_pipeline.pkl'
    path = f'models/{filename}'
    joblib.dump(pipe, path)
    size_kb = os.path.getsize(path) / 1024
    print(f'Saved: {path:55s}  ({size_kb:.1f} KB)')

In [ ]:
# Identify the best model by F1-Score
best_model_name = results_df.iloc[0]['Model']
best_pipeline   = trained_pipelines[best_model_name]

# Also save it separately as the 'production' model
joblib.dump(best_pipeline, 'models/best_model_pipeline.pkl')
print(f'Best model: {best_model_name}')
print(f'Saved as  : models/best_model_pipeline.pkl')
print(f'F1-Score  : {results_df.iloc[0]["F1-Score"]}%')

## 10. Verify: Load & Predict with the Saved Model

This is the most important step — confirm the saved `.pkl` works exactly like the in-memory pipeline.

In [ ]:
# Load from disk — completely fresh, no reliance on in-memory objects
loaded_model = joblib.load('models/best_model_pipeline.pkl')
print(f'Loaded model type: {type(loaded_model.named_steps["clf"]).__name__}\n')

# Test reviews
test_reviews = [
    "this product is absolutely amazing, works perfectly, love it!",
    "best buy ever great perfect wonderful amazing highly recommend",
    "the quality is okay but delivery was a bit slow",
    "does not work at all, very disappointed, complete waste of money",
]

predictions = loaded_model.predict(test_reviews)

print(f'{'Review':60s}  Prediction')
print('-' * 75)
for review, pred in zip(test_reviews, predictions):
    label = '🚩 FAKE (CG)' if pred == 'CG' else '✅ GENUINE (OR)'
    print(f'{review[:58]:60s}  {label}')

## 11. What to do with these files

After running this notebook you will have:

```
models/
├── best_model_pipeline.pkl          ← The model your dashboard loads
├── logistic_regression_pipeline.pkl
├── support_vector_machine_pipeline.pkl
├── random_forest_pipeline.pkl
├── decision_tree_pipeline.pkl
├── naive_bayes_pipeline.pkl
├── knn_k5_pipeline.pkl
└── model_comparison.png             ← Chart for your portfolio README
```

Your **Plotly Dash dashboard** (`app.py`) loads `best_model_pipeline.pkl` with:
```python
import joblib
model = joblib.load('models/best_model_pipeline.pkl')
pred = model.predict([user_input_text])[0]
```